# End-to-end trainig

In [ ]:
import torch

N_SRC = 10
BATCH_SIZE = 128
N_LCS = 10_000_000

DP1_ROOT = "../../data/dp1"
# DP1_ROOT = "/astro/store/epyc3/data3/hats/catalogs/dp1"
LSDB_WORKERS = 12
# DEVICE = "cpu"
# DEVICE = torch.device("cuda", 1)
DEVICE = "mps"

PLOT_MAGS = [18, 21, 25]

from uncle_val.pipelines.splits import dp1_config

survey_config = dp1_config(catalog_root=DP1_ROOT, n_src=N_SRC)

In [ ]:
from uncle_val.learning.losses import (
    minus_ln_chi2_prob_loss,
    kl_divergence_whiten_loss,
    epps_pulley_whiten_loss,
)
from uncle_val.pipelines import run_rubin_dp_constant_magerr

model_path = run_rubin_dp_constant_magerr(
    band="r",
    non_extended_only=True,
    n_workers=LSDB_WORKERS,
    n_lcs=N_LCS,
    loss_fn=epps_pulley_whiten_loss(lmbd=None, soft=None, kind="accum"),
    val_losses={
        "Total Soften KL": kl_divergence_whiten_loss(soft=20.0, kind="accum", lmbd=None),
        "Total Soften -ln(p_χ²)": minus_ln_chi2_prob_loss(soft=20.0, kind="accum", lmbd=None),
    },
    train_batch_size=BATCH_SIZE,
    val_batch_size=4098,
    start_tfboard=True,
    output_root="./runs",
    device=DEVICE,
    survey_config=survey_config,
)

In [10]:
# model_path = "/Users/hombit/projects/lincc-frameworks/uncle-val/docs/pre_executed/runs/2026-02-17_10-46/constant_magerr.pt"

print(model_path)
model = torch.load(model_path, weights_only=False)

for name, param in model.named_parameters():
    print(name, param)

/Users/hombit/projects/lincc-frameworks/uncle-val/docs/pre_executed/runs/2026-02-17_10-46/constant_magerr.pt
addition_centi_mag_err Parameter containing:
tensor([1.9615], device='mps:0', requires_grad=True)


In [17]:
from uncle_val.learning.models import ConstantMagErrModel

model_path = "/tmp/model.pt"
new_model = ConstantMagErrModel(input_names=["x", "err"])
new_model.addition_centi_mag_err = torch.nn.Parameter(torch.tensor(1.0))
torch.save(new_model, model_path)

### Train metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="train",
    survey_config=survey_config,
    non_extended_only=False,
    n_workers=LSDB_WORKERS,
    model_path=model_path,
    model_columns=["lc.x", "lc.err"],
    device=DEVICE,
    n_samples=5,
    object_mags=[18, 21, 25],
)

### Validation metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="val",
    survey_config=survey_config,
    non_extended_only=False,
    n_workers=LSDB_WORKERS,
    model_path=model_path,
    model_columns=["lc.x", "lc.err"],
    # model_path="runs/2025-10-10_20-39/mlp.pt",
    # model_columns=['lc.x', 'lc.err', 'extendedness', 'is_u_band', 'is_g_band', 'is_r_band', 'is_i_band', 'is_z_band', 'is_y_band'],
    device=DEVICE,
    n_samples=5,
    object_mags=[18, 21, 25],
)

### Test metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="test",
    survey_config=survey_config,
    non_extended_only=False,
    n_workers=LSDB_WORKERS,
    model_path=model_path,
    model_columns=["lc.x", "lc.err"],
    # model_path="./runs/2025-11-21_16-56/mlp.pt",
    # model_columns=['lc.x', 'lc.err', 'extendedness', 'lc.skyBg', 'lc.seeing', 'lc.expTime', 'is_u_band', 'is_g_band', 'is_r_band', 'is_i_band', 'is_z_band', 'is_y_band'],
    device=DEVICE,
    n_samples=5,
    object_mags=[18, 21, 25],
)